# Agent 1 — Real Agentic Retrieval with LangGraph + arXiv

This notebook is the first main workshop notebook.

Its goal is to teach the core mechanics of an agent workflow by building a **small but real** research agent with:

- explicit state,
- plain Python node functions,
- a real public retrieval tool,
- conditional routing,
- a visible loop,
- and a clear stopping condition.

We will keep the code beginner-friendly, but this is **not** a toy chatbot.  
It is a workflow that decides when to retrieve, gathers evidence, and only then produces an answer.


## What this notebook is trying to teach

By the end of this notebook, students should be able to explain:

1. why state is the backbone of an agent,
2. how node functions divide the work,
3. how tools fit into the workflow,
4. how routing decisions create loops,
5. and why stopping conditions matter.

We also keep one practical rule in mind throughout the notebook:

> Tools are capabilities, not intelligence.  
> The graph decides **when** to use them.  
> The tool only does the action.


In [ ]:
%pip install -q langgraph langchain langchain-ollama pydantic requests langsmith python-dotenv


In [ ]:
from __future__ import annotations

from typing import Literal, Optional, TypedDict
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

import requests
import xml.etree.ElementTree as ET
from textwrap import shorten


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "workshop_02"

In [ ]:
USE_REAL_LLM = True
USE_REAL_TOOLS = True
USE_STRUCTURED_OUTPUT = True
USE_CHECKPOINTER = False
USE_STREAMING_PREVIEW = False

OLLAMA_MODEL = "gemma4:e4b"
ARXIV_MAX_RESULTS = 4
ARXIV_TIMEOUT_SECONDS = 20


## Local model setup

The workshop uses **ChatOllama** for all LLM calls.

For class, it is useful to keep both a **real** mode and a **deterministic fallback** mode:

- `USE_REAL_LLM=True` runs the local model.
- `USE_REAL_LLM=False` keeps the notebook usable even if Ollama is not available.
- `USE_REAL_TOOLS=False` keeps the notebook usable even if the network is down.


In [ ]:
llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
    num_ctx=10000,
    reasoning=True,
)

def call_llm_text(messages: list) -> str:
    response = llm.invoke(messages)
    print(f"{'*'*20}\nLLM Response:\n{response}\n{'*'*20}")
    return response.content if hasattr(response, "content") else str(response)

def call_llm_structured(messages: list, schema: type[BaseModel]):
    structured_llm = llm.with_structured_output(schema, include_raw=True)
    result = structured_llm.invoke(messages)
    print(f"{'*'*20}\nLLM Response:\n{result}\n{'*'*20}")

    parsed = result.get("parsed")
    if parsed is None:
        raw = result.get("raw")
        raise ValueError(f"Structured output parsing failed. Raw response: {raw}")

    return parsed

checkpointer = InMemorySaver() if USE_CHECKPOINTER else None


## The research question

This agent is intentionally focused on the same teaching topic used in the workshop draft:

- **agentic RAG**
- **corrective RAG**

That gives us a real research-style question, while still keeping the domain small enough for a classroom walkthrough.


In [ ]:
QUESTION_EXAMPLES = [
    "What themes appear in papers about agentic RAG and corrective RAG?",
    "How do recent papers describe retrieval quality control in corrective RAG?",
    "What roles do grading and query rewriting play in agentic RAG workflows?",
]

for i, question in enumerate(QUESTION_EXAMPLES, start=1):
    print(f"{i}. {question}")


## State design

The state is the shared memory of the graph.

It stores both the **task input** and the **intermediate working data** produced by the nodes.


In [ ]:
class ResearchState(TypedDict):
    question: str
    findings: list[str]
    search_queries: list[str]
    last_decision: Optional[str]
    enough_information: bool
    iteration: int
    max_iterations: int
    final_answer: Optional[str]
    errors: list[str]


## Structured decision output

We do **not** want to parse a vague paragraph from the model if we can avoid it.

So we use a small Pydantic schema for the decision step.


In [ ]:
class DecisionOutput(BaseModel):
    action: Literal["search", "finalize"]
    reasoning: str = Field(description="Short explanation of the decision.")
    query: Optional[str] = Field(
        default=None,
        description="Focused paper-search query if action is search.",
    )


## Real retrieval tool: arXiv

The first agent uses the **arXiv API** as a real public source.

This keeps the example realistic without requiring an API key.

We also add a lightweight deterministic fallback corpus aligned to the same topic, so the notebook remains teachable even if:
- the network is unavailable,
- the API is temporarily slow,
- or the model is not running locally.


In [ ]:
RAG_FALLBACK_CORPUS = {
    "agentic rag": [
        {
            "title": "Fallback note: Agentic RAG as a workflow pattern",
            "summary": (
                "Agentic RAG often treats retrieval as one step inside a broader workflow. "
                "The model may decide whether to search, whether retrieved evidence is useful, "
                "and whether to refine the query before answering."
            ),
            "link": "fallback://agentic-rag-overview",
        },
        {
            "title": "Fallback note: Iterative retrieval",
            "summary": (
                "A recurring theme is iterative retrieval: search, inspect results, reformulate, "
                "and only then synthesize an answer."
            ),
            "link": "fallback://iterative-retrieval",
        },
    ],
    "corrective rag": [
        {
            "title": "Fallback note: Corrective RAG",
            "summary": (
                "Corrective RAG introduces a quality-control step that checks whether retrieved "
                "material is relevant enough. Weak retrieval can trigger query rewriting or another search."
            ),
            "link": "fallback://corrective-rag-overview",
        },
        {
            "title": "Fallback note: Retrieval grading",
            "summary": (
                "Corrective workflows often separate retrieval from validation. The system does not blindly trust "
                "documents just because they were found."
            ),
            "link": "fallback://retrieval-grading",
        },
    ],
    "query rewriting rag": [
        {
            "title": "Fallback note: Query rewriting in RAG",
            "summary": (
                "Query rewriting is a practical bridge between poor first-pass retrieval and a better second attempt."
            ),
            "link": "fallback://query-rewriting",
        }
    ],
}

def arxiv_search_tool(query: str, max_results: int = ARXIV_MAX_RESULTS) -> list[dict]:
    url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": f"all:{query}",
        "start": 0,
        "max_results": max_results,
    }
    headers = {
        "User-Agent": "Workshop-Agent/1.0 (educational notebook example)"
    }

    response = requests.get(url, params=params, headers=headers, timeout=ARXIV_TIMEOUT_SECONDS)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    ns = {"atom": "http://www.w3.org/2005/Atom"}

    papers = []
    for entry in root.findall("atom:entry", ns):
        title = (entry.findtext("atom:title", default="", namespaces=ns) or "").strip()
        summary = (entry.findtext("atom:summary", default="", namespaces=ns) or "").strip()
        link = (entry.findtext("atom:id", default="", namespaces=ns) or "").strip()

        authors = []
        for author in entry.findall("atom:author", ns):
            name = (author.findtext("atom:name", default="", namespaces=ns) or "").strip()
            if name:
                authors.append(name)

        papers.append(
            {
                "title": title,
                "summary": summary,
                "link": link,
                "authors": authors,
            }
        )

    return papers

def fallback_search_tool(query: str) -> list[dict]:
    normalized = query.strip().lower()
    if normalized in RAG_FALLBACK_CORPUS:
        return RAG_FALLBACK_CORPUS[normalized]

    if "corrective" in normalized:
        return RAG_FALLBACK_CORPUS["corrective rag"]
    if "rewrite" in normalized:
        return RAG_FALLBACK_CORPUS["query rewriting rag"]
    return RAG_FALLBACK_CORPUS["agentic rag"]


## Deterministic vs model-driven control

This notebook deliberately shows **both**:

- a deterministic controller,
- and an LLM-based controller with structured output.

That is an important engineering lesson.

A graph should still be understandable when the model is removed.


In [ ]:
def choose_action_deterministic(state: ResearchState) -> DecisionOutput:
    if state["iteration"] == 0:
        return DecisionOutput(
            action="search",
            reasoning="We need a first retrieval pass to gather papers.",
            query="agentic rag",
        )

    if len(state["findings"]) < 3 and state["iteration"] < state["max_iterations"]:
        next_query = "corrective rag" if "corrective rag" not in state["search_queries"] else "query rewriting rag"
        return DecisionOutput(
            action="search",
            reasoning="We still need one more focused retrieval pass.",
            query=next_query,
        )

    return DecisionOutput(
        action="finalize",
        reasoning="We already have enough evidence to write a short grounded answer.",
        query=None,
    )


In [ ]:
def choose_action_llm(state: ResearchState) -> DecisionOutput:
    messages = [
        SystemMessage(content=(
            "You are controlling a small research workflow.\n\n"
            "Decide whether the workflow should:\n"
            "- search for more papers, or\n"
            "- finalize the answer.\n\n"
            "Rules:\n"
            "- Prefer short topic-style arXiv search queries.\n"
            "- Create the next search query based the asked questions and the gaps between current findings.\n"
            "- If there is already enough evidence, choose \"finalize\".\n"
            "- Return structured output only."
        )),
        HumanMessage(content=(
            f"Question:\n{state['question']}\n\n"
            f"Queries already tried:\n{state['search_queries']}\n\n"
            f"Current findings:\n{state['findings']}\n\n"
            f"Current iteration:\n{state['iteration']} / {state['max_iterations']}"
        )),
    ]
    return call_llm_structured(messages, DecisionOutput)

def choose_action(state: ResearchState) -> DecisionOutput:
    if not (USE_REAL_LLM and USE_STRUCTURED_OUTPUT):
        return choose_action_deterministic(state)

    return choose_action_llm(state)


## A deterministic answer fallback

We also want a non-LLM fallback for the final summarization step.


In [ ]:
def deterministic_summary(findings: list[str]) -> str:
    if not findings:
        return "No findings were collected, so the agent cannot yet summarize the topic."

    bullets = []
    for item in findings[:4]:
        bullets.append(f"- {shorten(item, width=160, placeholder='...')}")

    return (
        "Draft answer based on the collected findings:\n\n"
        "Common themes include iterative retrieval, explicit quality checks on retrieved evidence, "
        "and corrective steps such as query rewriting or another search pass when the first retrieval is weak.\n\n"
        "Evidence snippets:\n"
        + "\n".join(bullets)
    )


## Node functions

Each node should do one narrow job.

That keeps the workflow readable and debuggable.


In [ ]:
def initialize_research(state: ResearchState) -> dict:
    return {
        "findings": [],
        "search_queries": [],
        "last_decision": None,
        "enough_information": False,
        "iteration": 0,
        "final_answer": None,
        "errors": [],
    }

def decide_next_step(state: ResearchState) -> dict:
    errors = list(state["errors"])

    try:
        decision = choose_action(state)
        if decision is None:
            raise ValueError("Decision model returned no parsed structured output.")
    except Exception as exc:
        errors.append(f"LLM decision fallback triggered: {exc}")
        decision = choose_action_deterministic(state)
    
    updates = {
        "iteration": state["iteration"] + 1,
        "last_decision": decision.reasoning,
        "errors": errors,
    }

    if decision.query:
        updates["search_queries"] = state["search_queries"] + [decision.query]

    if decision.action == "finalize":
        updates["enough_information"] = True

    return updates

def run_search(state: ResearchState) -> dict:
    query = state["search_queries"][-1]

    findings = []
    errors = list(state["errors"])

    if USE_REAL_TOOLS:
        try:
            papers = arxiv_search_tool(query)
        except Exception as exc:
            errors.append(f"arXiv search failed for query '{query}': {exc}")
            papers = fallback_search_tool(query)
    else:
        papers = fallback_search_tool(query)

    for paper in papers:
        authors = ", ".join(paper.get("authors", [])) if paper.get("authors") else "Unknown authors"
        findings.append(
            "\n".join(
                [
                    f"Title: {paper['title']}",
                    f"Authors: {authors}",
                    f"Summary: {paper['summary']}",
                    f"Link: {paper['link']}",
                ]
            )
        )

    return {
        "findings": state["findings"] + findings,
        "errors": errors,
    }

def assess_information(state: ResearchState) -> dict:
    enough = len(state["findings"]) >= 3 or state["iteration"] >= state["max_iterations"]
    return {"enough_information": enough}

def finalize_research(state: ResearchState) -> dict:
    errors = list(state["errors"])

    if USE_REAL_LLM:
        messages = [
            SystemMessage(content=(
                "Write a short informative answer to the question below.\n\n"
                "Requirements:\n"
                "- keep it short,\n"
                "- mention patterns rather than copying the findings,\n"
                "- stay grounded in the retrieved evidence,\n"
                "- mention at least two recurring ideas if possible."
            )),
            HumanMessage(content=(
                f"Question:\n{state['question']}\n\n"
                f"Collected findings:\n{state['findings']}"
            )),
        ]
        try:
            answer = call_llm_text(messages)
        except Exception as exc:
            errors.append(f"LLM finalization fallback triggered: {exc}")
            answer = deterministic_summary(state["findings"])
    else:
        answer = deterministic_summary(state["findings"])

    return {
        "final_answer": answer,
        "errors": errors,
    }


## Build the graph

This is the key architectural moment.

Read the graph as a workflow, not as a bag of helper functions.


In [ ]:
graph_builder = StateGraph(ResearchState)

graph_builder.add_node("initialize", initialize_research)
graph_builder.add_node("decide_next_step", decide_next_step)
graph_builder.add_node("run_search", run_search)
graph_builder.add_node("assess_information", assess_information)
graph_builder.add_node("finalize", finalize_research)

graph_builder.add_edge(START, "initialize")
graph_builder.add_edge("initialize", "decide_next_step")

def route_after_decision(state: ResearchState) -> str:
    if state["enough_information"]:
        return "finalize"
    return "run_search"

graph_builder.add_conditional_edges(
    "decide_next_step",
    route_after_decision,
    {
        "run_search": "run_search",
        "finalize": "finalize",
    },
)

graph_builder.add_edge("run_search", "assess_information")

def route_after_assessment(state: ResearchState) -> str:
    if state["enough_information"]:
        return "finalize"
    return "decide_next_step"

graph_builder.add_conditional_edges(
    "assess_information",
    route_after_assessment,
    {
        "decide_next_step": "decide_next_step",
        "finalize": "finalize",
    },
)

graph_builder.add_edge("finalize", END)

research_graph = graph_builder.compile(
    checkpointer=checkpointer if USE_CHECKPOINTER else None
)

## Run the graph with `invoke()`

`invoke()` is the simplest way to run the whole workflow from start to finish.


In [ ]:
initial_state: ResearchState = {
    "question": "What themes appear in papers about agentic RAG and corrective RAG?",
    "findings": [],
    "search_queries": [],
    "last_decision": None,
    "enough_information": False,
    "iteration": 0,
    "max_iterations": 3,
    "final_answer": None,
    "errors": [],
}

result = research_graph.invoke(initial_state)
result


## Inspect the result

This is part of observability.

We want to see:
- which queries were tried,
- what evidence was stored,
- and what answer the graph produced.


In [ ]:
print("Queries tried:")
for query in result["search_queries"]:
    print("-", query)

print("\nLast decision:")
print(result["last_decision"])

print("\nCollected findings:")
for i, item in enumerate(result["findings"], start=1):
    print(f"\nFinding {i}:")
    print(item[:800])

print("\nFinal answer:")
print(result["final_answer"])

print("\nErrors:")
for error in result["errors"]:
    print("-", error)


## Optional streaming preview

We will explain streaming properly in Appendix 91.

For now, this is just a preview that a graph does not have to run as one silent black box.


In [ ]:
if USE_STREAMING_PREVIEW:
    for chunk in research_graph.stream(initial_state, stream_mode="updates"):
        print(chunk)


## Why this already counts as an agent workflow

Even this small notebook already contains the core ideas:

- explicit state,
- explicit routing,
- plain Python nodes,
- a reusable real tool,
- a loop,
- and a stopping condition.

That is enough to teach the shape of an agent without hiding the mechanics.


## Bridge to the main assignment

This notebook is small, but the shape already matches the larger course project:

- decide what to do next,
- call tools,
- validate what came back,
- loop when needed,
- and stop when the workflow has enough information.

The next notebook keeps the same philosophy, but moves to a more realistic security collection workflow.
